In [10]:
import pandas as pd, numpy as np, sklearn, xgboost, lifelines, shap
import matplotlib.pyplot as plt
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("xgboost:", xgboost.__version__)
print("lifelines:", lifelines.__version__)

pandas: 2.3.3
numpy: 2.4.6
xgboost: 3.3.0
lifelines: 0.30.3


In [11]:
preview = pd.read_csv('data/accepted_2007_to_2018Q4.csv.gz', compression='gzip', nrows=5)
print(preview.shape)
preview.head()

(5, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
columns = preview.columns
for col in columns:
    print(col)


id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

In [13]:
cols_needed = ['id', 'issue_d', 'term', 'grade', 'sub_grade', 'loan_status']
df_status = pd.read_csv('data/accepted_2007_to_2018Q4.csv.gz', compression='gzip', usecols=cols_needed)
print(df_status.shape)
df_status['loan_status'].value_counts()

C:\Users\Rashtra Mitra\AppData\Local\Temp\ipykernel_7652\3223788720.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_status = pd.read_csv('data/accepted_2007_to_2018Q4.csv.gz', compression='gzip', usecols=cols_needed)


(2260701, 6)


loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [14]:
good_statuses = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']
bad_statuses  = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off', 'Default']

resolved_mask = df_status['loan_status'].isin(good_statuses + bad_statuses)
df_resolved = df_status[resolved_mask].copy()
df_resolved['target'] = df_resolved['loan_status'].isin(bad_statuses).astype(int)

print(f"Resolved loans: {len(df_resolved):,} of {len(df_status):,} total ({len(df_resolved)/len(df_status)*100:.1f}%)")
print(df_resolved['target'].value_counts(normalize=True).rename({0:'good',1:'bad'}))

Resolved loans: 1,348,099 of 2,260,701 total (59.6%)
target
good    0.800193
bad     0.199807
Name: proportion, dtype: float64


In [15]:
df_resolved['issue_d'] = pd.to_datetime(df_resolved['issue_d'], format='%b-%Y')
print(df_resolved['issue_d'].min(), "→", df_resolved['issue_d'].max())
df_resolved['issue_d'].dt.year.value_counts().sort_index()

2007-06-01 00:00:00 → 2018-12-01 00:00:00


issue_d
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134804
2014    223103
2015    375546
2016    293105
2017    169321
2018     56318
Name: count, dtype: int64

In [16]:
vintage_summary = df_resolved.groupby(df_resolved['issue_d'].dt.year).agg(
    n_loans=('target', 'size'),
    bad_rate=('target', 'mean')
)
print(vintage_summary)

         n_loans  bad_rate
issue_d                   
2007         603  0.262023
2008        2393  0.207271
2009        5281  0.136906
2010       12537  0.140145
2011       21721  0.151789
2012       53367  0.161973
2013      134804  0.155960
2014      223103  0.184498
2015      375546  0.201850
2016      293105  0.232859
2017      169321  0.231330
2018       56318  0.157570


In [17]:
df_status['issue_d'] = pd.to_datetime(df_status['issue_d'], format='%b-%Y')

total_by_year = df_status.groupby(df_status['issue_d'].dt.year).size()
resolved_by_year = df_resolved.groupby(df_resolved['issue_d'].dt.year).size()

maturity = pd.DataFrame({'total_originated': total_by_year, 'resolved': resolved_by_year})
maturity['pct_resolved'] = maturity['resolved'] / maturity['total_originated']
print(maturity)

         total_originated  resolved  pct_resolved
issue_d                                          
2007.0                603       603      1.000000
2008.0               2393      2393      1.000000
2009.0               5281      5281      1.000000
2010.0              12537     12537      1.000000
2011.0              21721     21721      1.000000
2012.0              53367     53367      1.000000
2013.0             134814    134804      0.999926
2014.0             235629    223103      0.946840
2015.0             421095    375546      0.891832
2016.0             434407    293105      0.674724
2017.0             443579    169321      0.381716
2018.0             495242     56318      0.113718


In [18]:
TRAIN_YEARS = list(range(2007, 2015))  # 2007–2014 inclusive
TEST_YEARS = [2015]

train_df = df_resolved[df_resolved['issue_d'].dt.year.isin(TRAIN_YEARS)].copy()
test_df  = df_resolved[df_resolved['issue_d'].dt.year.isin(TEST_YEARS)].copy()

print(f"Train: {len(train_df):,} loans | bad rate = {train_df['target'].mean():.3f}")
print(f"Test:  {len(test_df):,} loans | bad rate = {test_df['target'].mean():.3f}")

Train: 453,809 loans | bad rate = 0.170
Test:  375,546 loans | bad rate = 0.202
